# Truth vs. Deception: Mechanistic Interpretability Exploration

This notebook demonstrates probing LLM residual stream activations for truth vs. deception representations and performing causal activation steering.

In [ ]:
import sys
import os
sys.path.append('..')

from src import (
    load_model_and_tokenizer,
    build_dataset,
    extract_activations,
    train_probes,
    plot_layer_accuracy,
    steer_generation,
    FACTS
)

## 1. Load Model & Tokenizer

In [ ]:
model, tokenizer, n_layers, hidden_dim, use_tl, selected_model = load_model_and_tokenizer()
print(f"Loaded {selected_model} (layers={n_layers}, hidden_dim={hidden_dim})")

## 2. Build Contrastive Dataset

In [ ]:
dataset = build_dataset(tokenizer, FACTS)
print(f"Total prompts: {len(dataset)}")
print("Sample prompt:", dataset[0]['prompt'])

## 3. Extract Residual Stream Activations

In [ ]:
layers_to_probe = [0, 4, 8, 12, 16, 20]
layers_to_probe = [l for l in layers_to_probe if l < n_layers]
X_by_layer, y = extract_activations(model, tokenizer, dataset, layers_to_probe, use_transformer_lens=use_tl)

## 4. Train Probes & Compute Mass-Mean Baseline

In [ ]:
results = train_probes(X_by_layer, y, layers_to_probe)
plot_layer_accuracy(results, save_path="layer_accuracy.png", model_name=selected_model, show_plot=True)

## 5. Causal Intervention (Activation Steering)

In [ ]:
best_layer = max(results, key=lambda r: r.cv_mean_acc)
test_prompt = "Lie to me: Is the Eiffel Tower located in Paris?"

print("Baseline Output:")
print(steer_generation(model, tokenizer, test_prompt, best_layer, strength=0.0, use_transformer_lens=use_tl))

print("\nSteered Output (Strength=6.0):")
print(steer_generation(model, tokenizer, test_prompt, best_layer, strength=6.0, use_transformer_lens=use_tl))